# E-Commerce Sales Analysis

## Project Overview

This project analyzes an e-commerce dataset from Brazilian marketplace **Olist**.

The objective of this analysis is to explore sales performance and generate insights that could help improve business decisions.

Through this analysis we will answer questions such as:
- How have sales envolved over time?
- Wich product categories generate the most reveue?
- Where are costumers located?
- Wich products sell the most?

This project demostrates core Data Analyst skills including:
- Data cleaning
- Exploratory Data Analysis (EDA)
- Data transformation
- Data visualization
- Business insight generation 

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Loading the Data

The dataset consists of multiple tables representing diferent entities in the e-commerce platform.

The main table used in this analysis are:
- **orders** -> information about each order
- **order_items** -> products included in each order
- **products** -> product information
- **customers** -> customer information

This tables are connected through unique identifiers such as `order_id`, `product_id`, and `customer_id`.

In [4]:
orders = pd.read_csv("../data/raw/orders.csv")
order_items = pd.read_csv("../data/raw/order_items.csv")
products = pd.read_csv("../data/raw/products.csv")
customers = pd.read_csv("../data/raw/customers.csv")

## Previewing the Data

Before performing any analysis, it is important to inspect the datasets

This step helps us understand:
- The structure of the data
- Column names
- Potential missing values
- Data types

In [12]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


## Dataset dimensions

Understanding the size of each dataset helps estimate the scale of the analysis

In [13]:
orders.shape
order_items.shape
products.shape
customers.shape

(99441, 5)

## Checking Missing Values

Before starting the analysis, it is important to verify whether the dataset contains missing values.

Missing data can affect calculations and visuallizations, so identifying them early helps ensure the reliability of the analysis.

In [17]:
orders.isnull().sum()

order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

## Cheking for Duplicate Records

Duplicate rows can distort the analysis by counting the same event multiple times.

Therefore, we check if duplicate records exist in the dataset.

In [19]:
orders.duplicated().sum()

np.int64(0)

## Exploring the Orders Dataset

The `orders` table contains information about each order placed in the platform.

Key columns include:
- `order_id` → unique identifier for each order
- `customer_id` → identifies the customer who placed the order
- `order_purchase_timestamp` → time when the purchase occurred
- `order_status` → status of the order

In [5]:
orders["order_status"].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

## Converting Date Columns

To perform time-based analysis, we convert the purchase timestamp into datetime format.

This allows us to extract information such as:
- year
- month
- day

In [29]:
orders["order_purchase_timestamp"] = pd.to_datetime(orders["order_purchase_timestamp"])

## Filtering Completed Orders

For the purpose of sales analysis, we will focus only on orders that were successfully delivered

Orders with other statuses such as canceled or unavailable do not represent completed sales and therefore do not generate revenue.

In [30]:
delivered_orders = orders[orders["order_status"] == "delivered"]

In [31]:
delivered_orders.shape

(96478, 8)

## Explore Order Items

The `order_items` table contains information about the products included in each order.


Key columns include:

- `order_id`
- `product_id`
- `price`
- `freight_value`

Each row represents a product within an order

In [32]:
order_items.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [33]:
order_items.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  object 
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  object 
 3   seller_id            112650 non-null  object 
 4   shipping_limit_date  112650 non-null  object 
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), object(4)
memory usage: 6.0+ MB


## Merging Orders and Order Items

To analyse sales revenue, we need to combine the `orders` dataset with the `order_items` dataset.

The connection between this tables is the `order_id`.

This merge allows us to associate each product purchase with its coresponding order.

In [34]:
sales = pd.merge(
    delivered_orders,
    order_items,
    on="order_id",
    how="inner"
)

In [35]:
sales.shape
sales.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


## Understanding revenue

The `price` column represents the value of each product purchased.

Since each row represents a single product eithin an order, the total revenue can be calculated by summing the `price` column.

In [36]:
total_revenue = sales["price"].sum()

In [37]:
sales.shape

(110197, 14)

In [38]:
sales.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00,1,87285b34884572647811a353c7ac498a,3504c0cb71d7fa48d967e0e4c94d59d9,2017-10-06 11:07:15,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00,1,595fac2a385ac33a80bd5114aec74eb8,289cdb325fb7e7f891c38608bf9e0962,2018-07-30 03:24:27,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00,1,aa4383b373c6aca5d8797843e5594415,4869f7a5dfa277a7dca6462dcf3b52b2,2018-08-13 08:55:23,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00,1,d0b61bfb1de832b15ba9d266ca96e5b0,66922902710d126a0e7d26b0e3805106,2017-11-23 19:45:59,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00,1,65266b2da20d04dbe00c5c2d3bb7859e,2c9e548be18521d1c43cde1c582c6de8,2018-02-19 20:31:37,19.90,8.72


## Total Number of Orders

Another important metric is te total number of completed orders.

This metric helps understand the volume of transactions processed bu the platform.

In [39]:
total_orders = sales["order_id"].nunique()

## Average Order Value (AOV)

The Averacie Order Value represents the average amount spent per order.

This metric is widely used in e-commerce to understand customer purchasing behavior.


In [46]:
average_order_value = total_revenue / total_orders

## Extracting Purchase Month

To analyse sales trends over time, we extract the purchase month from the order timestamp.

In [42]:
sales["purchase_month"] = sales["order_purchase_timestamp"].dt.to_period("M")

In [43]:
total_revenue

np.float64(13221498.110000001)

In [44]:
total_orders


96478

In [47]:
average_order_value


np.float64(137.04158575011922)